# Query Expansion: Add Recall Without Losing Intent

| Field | Value |
|---|---|
| Stage | Query transformation |
| Difficulty | Intermediate |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
Expansion is a controlled vocabulary bridge. Every added term can recover a relevant document or introduce query drift.

## 30-Second Summary

This notebook compares a lexical baseline with curated and aggressive expansion on a four-document policy corpus. Curated terms bridge a paraphrase to the incident policy; an unrelated payment expansion demonstrates drift.

## Why This Matters

Users rarely repeat a document's exact terminology. Expansion can bridge aliases and acronyms, but unconstrained generation can change the question and retrieve confidently irrelevant context.

## Scope

| Covers | Does not cover |
|---|---|
| Synonym/alias expansion, deduplication, drift checks, top-1 measurement | Live LLM expansion, multilingual thesaurus, neural retrieval |


## Mental Model

```text
original query -> candidate terms -> deduplicate/limit -> retrieve -> compare with original intent
```


In [1]:
import re

def tokens(text: str) -> list[str]:
    stop = {"a", "an", "and", "are", "does", "for", "how", "is", "of", "the", "to", "what", "who"}
    return [token for token in re.findall(r"[a-z0-9]+", text.lower()) if token not in stop]

def overlap_rank(query: str, documents: list[dict]) -> list[dict]:
    query_terms = set(tokens(query))
    return sorted(
        documents,
        key=lambda document: (-len(query_terms & set(tokens(document["text"]))), document["id"]),
    )

documents = [
    {"id": "access", "text": "Role access requests require manager approval."},
    {"id": "billing", "text": "Invoice payment disputes receive a billing review."},
    {"id": "incidents", "text": "Priority one incidents are acknowledged within fifteen minutes."},
    {"id": "retention", "text": "Audit logs are retained for thirty days."},
]
query = "How quickly is a critical disruption recognized?"
expected_id = "incidents"


## How It Works

We normalize terms, add only approved aliases, and preserve the original query. Retrieval sees the union; evaluation still uses the original intent. Expansion size and source should be observable configuration.


## Baseline

The lexical baseline has no shared content term with the incident policy, so deterministic tie-breaking selects the wrong document.


In [2]:
baseline_ranking = overlap_rank(query, documents)
[(item["id"], len(set(tokens(query)) & set(tokens(item["text"])))) for item in baseline_ranking]


[('access', 0), ('billing', 0), ('incidents', 0), ('retention', 0)]

## Technique Implementation

The curated expansion maps `critical disruption` to `priority one incident` and `recognized` to `acknowledged`. Terms are deduplicated and capped; the mapping is inspectable rather than generated silently.


In [3]:
CURATED_ALIASES = {
    "critical": ["priority", "one"],
    "disruption": ["incident"],
    "recognized": ["acknowledged"],
}

def expand(query: str, aliases: dict[str, list[str]], limit: int = 5) -> str:
    additions = []
    for term in tokens(query):
        additions.extend(aliases.get(term, []))
    additions = list(dict.fromkeys(additions))[:limit]
    return " ".join([query, *additions])

curated_query = expand(query, CURATED_ALIASES)
curated_query, overlap_rank(curated_query, documents)[0]["id"]


('How quickly is a critical disruption recognized? priority one incident acknowledged',
 'incidents')

## Controlled Experiment

We compare the original, curated expansion, and an aggressive expansion that also adds unrelated billing terms. Hit@1 measures intent preservation on this single labeled query; expansion length is reported as a drift signal.


In [4]:
aggressive_aliases = {**CURATED_ALIASES, "quickly": ["invoice", "payment", "billing"]}
variants = {
    "original": query,
    "curated": curated_query,
    "aggressive": expand(query, aggressive_aliases, limit=8),
}
results = {
    name: {
        "top_id": overlap_rank(value, documents)[0]["id"],
        "hit@1": float(overlap_rank(value, documents)[0]["id"] == expected_id),
        "added_terms": len(set(tokens(value)) - set(tokens(query))),
    }
    for name, value in variants.items()
}
results


{'original': {'top_id': 'access', 'hit@1': 0.0, 'added_terms': 0},
 'curated': {'top_id': 'incidents', 'hit@1': 1.0, 'added_terms': 4},
 'aggressive': {'top_id': 'billing', 'hit@1': 0.0, 'added_terms': 7}}

## Evaluation

The original query misses. Curated expansion retrieves `incidents`; aggressive expansion drifts to `billing` because three unrelated added terms outweigh the intended bridge. This proves expansion needs caps, provenance, and regression labels.


In [5]:
assert results["original"]["hit@1"] == 0.0
assert results["curated"] == {"top_id": "incidents", "hit@1": 1.0, "added_terms": 4}
assert results["aggressive"]["top_id"] == "billing"
assert results["aggressive"]["added_terms"] > results["curated"]["added_terms"]
print("Query-expansion checks passed.")


Query-expansion checks passed.


## Decision Guide

| Need | Expansion source |
|---|---|
| Product aliases/acronyms | Governed domain dictionary |
| Spelling variants | Deterministic normalization |
| Broad discovery | Generated alternatives with caps and fusion |
| High precision | Original query plus conservative aliases |


## Failure Modes and Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Irrelevant topic dominates | Expansion drift | Cap terms, weight original, validate intent |
| Duplicate searches | Variants not normalized | Canonicalize and deduplicate |
| No gain | Corpus already shares vocabulary | Skip expansion by segment |
| Latency/cost spike | Too many variants | Parallelize with strict budget or fuse terms once |


## Production Notes

### Observability
Log original/added terms, expansion source/version, variant count, branch hits, and drift rate.

### Safety and Guardrails
Do not let expansion add unauthorized entities or bypass metadata filters.

### Latency and Cost
Cache deterministic expansions and cap generated variants before retrieval fan-out.


## Practice

Add one acronym and one ambiguous synonym. Define expected and forbidden document IDs before editing the alias map.

## Recall

Toggle - Recall: What is query drift?
Added terms change the original intent enough to favor irrelevant documents.

Toggle - Recall: Why keep the original query?
It anchors intent and provides a baseline/fusion branch.

## Sources

- [Query expansion survey](https://doi.org/10.1561/1500000010)
- Repository-owned synthetic policy fixture in this notebook

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for the controlled drift example | Evaluate governed aliases on the shared golden set |
